In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [12]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

merged = pd.read_csv('../data/all_merged.csv')
promotion = pd.read_csv('../data/promotion_df.csv')
normal = pd.read_csv('../data/normal_flow_df.csv')
final = pd.read_csv('../data/promotion_final2.csv')

---

In [13]:
display(final.head())
final.shape

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,flow_type,amount_from_received,amount_from_viewed,amount_raw
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,8.57,1,1,1,0,1,0,8.57,0.00,8.57
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,14.11,1,1,1,0,1,0,14.11,0.00,14.11
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,10.27,1,0,1,0,1,2,10.27,0.00,10.27
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,11.93,1,1,1,1,1,1,11.93,11.93,11.93
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,22.05,1,1,1,1,1,1,22.05,22.05,22.05


(76277, 28)

---

위의 표대로 계산하면 열람율 -> 전환율 -> 정상흐름비중 순으로 깔때기 모양의 퍼널이 나오게 되는 것 같다만 확실하진 않음<br>
생각해보니까 전환율과 정상흐름비중의 관계는 개념적으로 연결된 지표는 아님 -> 두 경우의 분모가 다르기 때문에

---

# 전체

In [14]:
# bogo, discount만 필터링
bd_df = final[final['offer_type'].isin(['bogo', 'discount'])].copy()

In [15]:
bd_df.shape

(61042, 28)

In [16]:
normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_funnel_dict(total, viewed, normal, invalid, completed_only):
    normal_viewed = total[total['flow_type'].isin([1, 3])]['is_viewed'].sum()
    
    return {
        'total_received': len(total),
        'total_viewed': total['is_viewed'].sum(),
        'total_completed': total['is_completed'].sum(),
        'normal_flow': len(normal),
        '열람율(%)': round(total['is_viewed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
        '정상열람율(%)': round(normal_viewed / len(total) * 100, 2) if len(total) > 0 else 0,
        '단계별전환율(%)': round(len(normal) / len(viewed) * 100, 2) if len(viewed) > 0 else 0,
        '정상흐름비중(%)': round(len(normal) / len(total) * 100, 2) if len(total) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid) / len(total) * 100, 2) if len(total) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only) / len(total) * 100, 2) if len(total) > 0 else 0,
        '완료율(%)': round(total['is_completed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
    }

# 전체
overall_funnel = pd.DataFrame([{
    'group': 'overall',
    **get_funnel_dict(bd_df, viewed_bd_df, normal_bd_df, invalid_bd_df, completed_only_bd_df)
}])
display(overall_funnel)

# offer_type
type_funnel_list = []
for offer_type, group in bd_df.groupby('offer_type'):
    viewed = viewed_bd_df[viewed_bd_df['offer_type'] == offer_type]
    normal = normal_bd_df[normal_bd_df['offer_type'] == offer_type]
    invalid = invalid_bd_df[invalid_bd_df['offer_type'] == offer_type]
    completed_only = completed_only_bd_df[completed_only_bd_df['offer_type'] == offer_type]
    type_funnel_list.append({
        'offer_type': offer_type,
        **get_funnel_dict(group, viewed, normal, invalid, completed_only)
    })

type_funnel = pd.DataFrame(type_funnel_list)
display(type_funnel)

# offer_id
id_funnel_list = []
for offer_id, group in bd_df.groupby('offer_id'):
    viewed = viewed_bd_df[viewed_bd_df['offer_id'] == offer_id]
    normal = normal_bd_df[normal_bd_df['offer_id'] == offer_id]
    invalid = invalid_bd_df[invalid_bd_df['offer_id'] == offer_id]
    completed_only = completed_only_bd_df[completed_only_bd_df['offer_id'] == offer_id]
    id_funnel_list.append({
        'offer_id': offer_id,
        **get_funnel_dict(group, viewed, normal, invalid, completed_only)
    })

id_funnel = pd.DataFrame(id_funnel_list)
display(id_funnel)

,group,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,overall,61042,46894,33101,23267,76.82,69.93,54.5,38.12,6.89,9.22,54.23


,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,bogo,30499,25449,15501,10941,83.44,75.85,47.29,35.87,7.59,7.36,50.82
1,discount,30543,21445,17600,12326,70.21,64.02,63.03,40.36,6.19,11.08,57.62


,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,bogo_10_10_5,7593,7298,3301,2739,96.11,89.90,40.13,36.07,6.22,1.19,43.47
1,bogo_10_10_7,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47
2,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29
3,bogo_5_5_7,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
4,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
5,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82
6,discount_20_5_10,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
7,discount_7_3_7,7646,7337,5102,4348,95.96,88.14,64.52,56.87,7.82,2.04,66.73


## 전체
- 

---

# 채널별

In [17]:
channels = ['web', 'email', 'mobile', 'social']

normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_channel_funnel_dict(total, viewed, normal, invalid, completed_only):
    normal_viewed = total[total['flow_type'].isin([1, 3])]['is_viewed'].sum()
    return {
        'total_received': len(total),
        'total_viewed': total['is_viewed'].sum(),
        'total_completed': total['is_completed'].sum(),
        'normal_flow': len(normal),
        '열람율(%)': round(total['is_viewed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
        '정상열람율(%)': round(normal_viewed / len(total) * 100, 2) if len(total) > 0 else 0,
        '단계별전환율(%)': round(len(normal) / len(viewed) * 100, 2) if len(viewed) > 0 else 0,
        '정상흐름비중(%)': round(len(normal) / len(total) * 100, 2) if len(total) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid) / len(total) * 100, 2) if len(total) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only) / len(total) * 100, 2) if len(total) > 0 else 0,
        '완료율(%)': round(total['is_completed'].sum() / len(total) * 100, 2) if len(total) > 0 else 0,
    }

# 채널 전체
channel_funnel_list = []
for channel in channels:
    total = bd_df[bd_df[channel] == 1]
    viewed = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal = normal_bd_df[normal_bd_df[channel] == 1]
    invalid = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    channel_funnel_list.append({
        'channel': channel,
        **get_channel_funnel_dict(total, viewed, normal, invalid, completed_only)
    })

channel_funnel = pd.DataFrame(channel_funnel_list)
display(channel_funnel)

# 채널 x offer_type
channel_type_list = []
for channel in channels:
    total_temp = bd_df[bd_df[channel] == 1]
    viewed_temp = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal_temp = normal_bd_df[normal_bd_df[channel] == 1]
    invalid_temp = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only_temp = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    for offer_type in total_temp['offer_type'].unique():
        total_group = total_temp[total_temp['offer_type'] == offer_type]
        viewed_group = viewed_temp[viewed_temp['offer_type'] == offer_type]
        normal_group = normal_temp[normal_temp['offer_type'] == offer_type]
        invalid_group = invalid_temp[invalid_temp['offer_type'] == offer_type]
        completed_only_group = completed_only_temp[completed_only_temp['offer_type'] == offer_type]
        channel_type_list.append({
            'channel': channel,
            'offer_type': offer_type,
            **get_channel_funnel_dict(total_group, viewed_group, normal_group, invalid_group, completed_only_group)
        })

channel_type_funnel = pd.DataFrame(channel_type_list)
display(channel_type_funnel)

# 채널 x offer_id
channel_id_list = []
for channel in channels:
    total_temp = bd_df[bd_df[channel] == 1]
    viewed_temp = viewed_bd_df[viewed_bd_df[channel] == 1]
    normal_temp = normal_bd_df[normal_bd_df[channel] == 1]
    invalid_temp = invalid_bd_df[invalid_bd_df[channel] == 1]
    completed_only_temp = completed_only_bd_df[completed_only_bd_df[channel] == 1]
    for offer_id in total_temp['offer_id'].unique():
        total_group = total_temp[total_temp['offer_id'] == offer_id]
        viewed_group = viewed_temp[viewed_temp['offer_id'] == offer_id]
        normal_group = normal_temp[normal_temp['offer_id'] == offer_id]
        invalid_group = invalid_temp[invalid_temp['offer_id'] == offer_id]
        completed_only_group = completed_only_temp[completed_only_temp['offer_id'] == offer_id]
        channel_id_list.append({
            'channel': channel,
            'offer_id': offer_id,
            **get_channel_funnel_dict(total_group, viewed_group, normal_group, invalid_group, completed_only_group)
        })

channel_id_funnel = pd.DataFrame(channel_id_list)
display(channel_id_funnel)

,channel,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,web,53384,40178,29466,20685,75.26,68.56,56.52,38.75,6.70,9.74,55.20
1,email,61042,46894,33101,23267,76.82,69.93,54.50,38.12,6.89,9.22,54.23
2,mobile,53374,44231,29788,21967,82.87,75.45,54.55,41.16,7.42,7.23,55.81
3,social,38065,35942,21530,17759,94.42,87.00,53.63,46.65,7.43,2.48,56.56


,channel,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,web,bogo,22841,18733,11866,8359,82.01,74.62,49.04,36.60,7.39,7.96,51.95
1,web,discount,30543,21445,17600,12326,70.21,64.02,63.03,40.36,6.19,11.08,57.62
2,email,bogo,30499,25449,15501,10941,83.44,75.85,47.29,35.87,7.59,7.36,50.82
3,email,discount,30543,21445,17600,12326,70.21,64.02,63.03,40.36,6.19,11.08,57.62
4,mobile,bogo,30499,25449,15501,10941,83.44,75.85,47.29,35.87,7.59,7.36,50.82
5,mobile,discount,22875,18782,14287,11026,82.11,74.91,64.34,48.20,7.20,7.06,62.46
6,social,bogo,22822,21278,11198,8835,93.23,85.72,45.16,38.71,7.52,2.83,49.07
7,social,discount,15243,14664,10332,8924,96.20,88.91,65.85,58.54,7.29,1.95,67.78


,channel,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
0,web,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29
1,web,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
2,web,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82
3,web,discount_7_3_7,7646,7337,5102,4348,95.96,88.14,64.52,56.87,7.82,2.04,66.73
4,web,bogo_5_5_7,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
5,web,discount_20_5_10,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
6,web,bogo_10_10_5,7593,7298,3301,2739,96.11,89.90,40.13,36.07,6.22,1.19,43.47
7,email,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29
8,email,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
9,email,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82


---

# 채널 조합별

In [18]:
# 채널 조합 컬럼 생성
def get_channel_combo(row):
    channels = []
    for ch in ['web', 'email', 'mobile', 'social']:
        if row[ch] == 1:
            channels.append(ch)
    return '+'.join(channels) if channels else 'none'

bd_df['channel_combo'] = bd_df.apply(get_channel_combo, axis=1)

normal_bd_df = bd_df[bd_df['flow_type'] == 1].copy()
viewed_bd_df = bd_df[bd_df['flow_type'].isin([1, 3])].copy()
invalid_bd_df = bd_df[bd_df['flow_type'] == 0].copy()
completed_only_bd_df = bd_df[bd_df['flow_type'] == 2].copy()

def get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group):
    normal_viewed = group[group['flow_type'].isin([1, 3])]['is_viewed'].sum()
    return {
        'total_received': len(group),
        'total_viewed': group['is_viewed'].sum(),
        'total_completed': group['is_completed'].sum(),
        'normal_flow': len(normal_group),
        '열람율(%)': round(group['is_viewed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
        '정상열람율(%)': round(normal_viewed / len(group) * 100, 2) if len(group) > 0 else 0,
        '단계별전환율(%)': round(len(normal_group) / len(viewed_group) * 100, 2) if len(viewed_group) > 0 else 0,
        '정상흐름비중(%)': round(len(normal_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '비정상흐름비중(%)': round(len(invalid_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '바로완료비중(%)': round(len(completed_only_group) / len(group) * 100, 2) if len(group) > 0 else 0,
        '완료율(%)': round(group['is_completed'].sum() / len(group) * 100, 2) if len(group) > 0 else 0,
    }

# 채널 조합 전체
combo_funnel_list = []
for combo, group in bd_df.groupby('channel_combo'):
    viewed_group = viewed_bd_df[viewed_bd_df['channel_combo'] == combo]
    normal_group = normal_bd_df[normal_bd_df['channel_combo'] == combo]
    invalid_group = invalid_bd_df[invalid_bd_df['channel_combo'] == combo]
    completed_only_group = completed_only_bd_df[completed_only_bd_df['channel_combo'] == combo]
    combo_funnel_list.append({
        'channel_combo': combo,
        **get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group)
    })

combo_funnel = pd.DataFrame(combo_funnel_list).sort_values('total_received', ascending=False)
display(combo_funnel)

# 채널 조합 x offer_type
combo_type_list = []
for (combo, offer_type), group in bd_df.groupby(['channel_combo', 'offer_type']):
    viewed_group = viewed_bd_df[(viewed_bd_df['channel_combo'] == combo) & (viewed_bd_df['offer_type'] == offer_type)]
    normal_group = normal_bd_df[(normal_bd_df['channel_combo'] == combo) & (normal_bd_df['offer_type'] == offer_type)]
    invalid_group = invalid_bd_df[(invalid_bd_df['channel_combo'] == combo) & (invalid_bd_df['offer_type'] == offer_type)]
    completed_only_group = completed_only_bd_df[(completed_only_bd_df['channel_combo'] == combo) & (completed_only_bd_df['offer_type'] == offer_type)]
    combo_type_list.append({
        'channel_combo': combo,
        'offer_type': offer_type,
        **get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group)
    })

combo_type_funnel = pd.DataFrame(combo_type_list).sort_values('total_received', ascending=False)
display(combo_type_funnel)

# 채널 조합 x offer_id
combo_id_list = []
for (combo, offer_id), group in bd_df.groupby(['channel_combo', 'offer_id']):
    viewed_group = viewed_bd_df[(viewed_bd_df['channel_combo'] == combo) & (viewed_bd_df['offer_id'] == offer_id)]
    normal_group = normal_bd_df[(normal_bd_df['channel_combo'] == combo) & (normal_bd_df['offer_id'] == offer_id)]
    invalid_group = invalid_bd_df[(invalid_bd_df['channel_combo'] == combo) & (invalid_bd_df['offer_id'] == offer_id)]
    completed_only_group = completed_only_bd_df[(completed_only_bd_df['channel_combo'] == combo) & (completed_only_bd_df['offer_id'] == offer_id)]
    combo_id_list.append({
        'channel_combo': combo,
        'offer_id': offer_id,
        **get_combo_funnel_dict(group, viewed_group, normal_group, invalid_group, completed_only_group)
    })

combo_id_funnel = pd.DataFrame(combo_id_list).sort_values('total_received', ascending=False)
display(combo_id_funnel)

,channel_combo,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
3,web+email+mobile+social,30407,29226,17895,15177,96.12,88.88,56.16,49.91,7.24,1.70,58.85
2,web+email+mobile,15309,8289,8258,4208,54.14,46.74,58.81,27.49,7.41,19.05,53.94
1,web+email,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
0,email+mobile+social,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47


,channel_combo,offer_type,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
5,web+email+mobile+social,discount,15243,14664,10332,8924,96.20,88.91,65.85,58.54,7.29,1.95,67.78
4,web+email+mobile+social,bogo,15164,14562,7563,6253,96.03,88.84,46.41,41.24,7.19,1.45,49.87
2,web+email+mobile,bogo,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
1,web+email,discount,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
0,email+mobile+social,bogo,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47
3,web+email+mobile,discount,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82


,channel_combo,offer_id,total_received,total_viewed,total_completed,normal_flow,열람율(%),정상열람율(%),단계별전환율(%),정상흐름비중(%),비정상흐름비중(%),바로완료비중(%),완료율(%)
2,web+email+mobile,bogo_5_5_7,7677,4171,4303,2106,54.33,46.53,58.96,27.43,7.80,20.82,56.05
1,web+email,discount_20_5_10,7668,2663,3313,1300,34.73,31.55,53.74,16.95,3.18,23.07,43.21
0,email+mobile+social,bogo_10_10_7,7658,6716,3635,2582,87.70,79.52,42.40,33.72,8.17,5.58,47.47
7,web+email+mobile+social,discount_7_3_7,7646,7337,5102,4348,95.96,88.14,64.52,56.87,7.82,2.04,66.73
3,web+email+mobile,discount_10_2_7,7632,4118,3955,2102,53.96,46.95,58.67,27.54,7.01,17.27,51.82
6,web+email+mobile+social,discount_10_2_10,7597,7327,5230,4576,96.45,89.69,67.16,60.23,6.75,1.86,68.84
4,web+email+mobile+social,bogo_10_10_5,7593,7298,3301,2739,96.11,89.90,40.13,36.07,6.22,1.19,43.47
5,web+email+mobile+social,bogo_5_5_5,7571,7264,4262,3514,95.95,87.78,52.87,46.41,8.16,1.72,56.29


| flow_type | 흐름 설명                     | 비고            |
|-----------|------------------------------|-----------------|
| 0         | 받기 → 완료 → 보기            | 비정상 흐름      |
| 1         | 받기 → 보기 → 완료            | 정상 흐름        |
| 2         | 받기 → NaN -> 완료                   | 바로 완료   |
| 3         | 받기 → 보기 -> NaN                   | 완료 없이 보기   |
| 4         | 받기 -> NaN -> NaN                        | 반응 없음        |

| 지표     | 분자                                   | 분모                                   |
|----------|----------------------------------------|----------------------------------------|
| 열람율   | flow_type.isin([1,3])의 is_viewed       | flow_type.isin([0,1,2,3,4])의 count      |
| 전환율   | flow_type.isin([1,3])의 is_completed    | flow_type.isin([0,1,2,3,4])의 count      |
| 단계별전환율 | flow_type.isin([1])의 is_completed | flow_type.isin([1,3])의 count |
| 완료율   | flow_type.isin([0,1,2,3,4])의 is_completed | flow_type.isin([0,1,2,3,4])의 count |
| 정상흐름비중 | flow_type == 1의 count | flow_type.isin([0,1,2,3,4])의 count |
| 비정상흐름비중 | flow_type == 0의 count | flow_type.isin([0,1,2,3,4])의 count |
| 바로완료비중 | flow_type == 2의 count | flow_type.isin([0,1,2,3,4])의 count |

1. 전체: recieved
2. 열람: 전체 중 `받기 -> 보기`가 있는 경우
3. 완료: 전체 중 `받기 -> 보기 -> 완료`가 있는 경우<br>
-> 완료율이 정상흐름 완료율과 같이짐
=> 퍼널 흐름?